# Discrete-time quantum walk on a complex network

Generate the source four-node graph, prepare the native position/coin state and run three reflection-and-shift rounds. The dropped coin remains a quantum environment; the public output is the position marginal.


In [ ]:
from examples.synthesis.tutorials.basics.quantumwalk_complex_network.overview.main import (
    archived_results, make_graph, neighbor_probabilities, plot_archived_results,
    plot_graph, plot_samples, run_workflow,
)

graph = make_graph()
print("Neighbor probabilities:", neighbor_probabilities(graph))
graph_figure = plot_graph()


## Native synthesis and local readout

The existing native qfunc allocates logical position and coin arrays, prepares neighbor probabilities, applies three conditional diffusers and swaps, and drops the coin. No physical wires or handwritten gates are needed here. The unspecified source cloud sampling is made reproducible as 1000 local shots with seed17.


The complete round structure is editable below. `prepare` loads each node's
neighbor probabilities; `coin` is the reusable controlled reflection defined
in [the native program](../main.py). Static `steps` is expanded at synthesis,
while the position and coin remain quantum. The compiler handles coherent
control and inverse regions inside those helpers.

`drop(y)` hides the entangled coin from returned registers; it does **not**
reset it to zero. Readout of x takes its marginal. This is different from
cleaning compiler work bits. `run_workflow` retains the original seeded
1000-shot readout and checks that graph generation still matches the bound
neighbor vectors. Try `steps=1` in the visible program and compare the exact
position probabilities; the archived hardware plots remain historical data.


In [ ]:
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import SWAP, Output, QArray, QBit, allocate, drop, phase, qfunc
from examples.synthesis.tutorials.basics.quantumwalk_complex_network.main import prepare, coin

@qfunc
def walk_program(steps: int, angle: float, x: Output[QArray[QBit]]) -> None:
    y: QArray[QBit] = QArray()
    allocate(2, x)
    allocate(2, y)
    prepare(x, y)
    phase(x[0], angle)
    phase(y[1], -0.7 * angle)
    for _ in range(steps):
        coin(x, y)
        SWAP(x[0], y[0])
        SWAP(x[1], y[1])
    drop(y)

visible_program = synth.synthesize(
    walk_program, classical_inputs={"steps": 3, "angle": 0.0},
    max_qubits=5, max_workspace=1, memory_limit_bytes=1024**2,
)
visible_probabilities = dict(visible_program.probabilities(fq.run(visible_program), registers=("x",)))
print("Named position probabilities:", visible_probabilities)
program, counts = run_workflow()


## Position probabilities

Each integer is a graph node, rather than an array-bit record. Plot the locally sampled marginal over all four nodes.


In [ ]:
sample_figure = plot_samples(counts)


## Historical hardware comparison

Retain the source four/eight-node ideal arrays and all Torino counts at times1..4. The five-panel plot uses the eight-node graph and historical data. It does not run a new hardware job or eight-node simulation. The source final cell swapped HW-aware/agnostic aliases; labels here follow the original count declarations.

Live hardware-aware compilation and execution require a supported provider, credentials and a separately accepted deployment path. The commented source examples do not certify one here.


In [ ]:
archived_four = archived_results(4)
archived_eight = archived_results(8)
print("Archived graph sizes:", len(archived_four["ideal"][0]), len(archived_eight["ideal"][0]))
archive_figure = plot_archived_results()
